# First scan in scanlang — Jupyter edition

Run end-to-end with the project's docs interpreter:

```sh
uv run jupyter nbconvert --execute --to notebook --inplace \
  docs/notebooks/01_first_scan.ipynb
```

Goal: from a tiny OHLCV frame to a scored, filtered, ordered
result — using the same fixture the `docs/examples/*.py` scripts
use, so docs and notebooks never drift.


In [1]:
import sys

# make _fixture importable when nbconvert runs from the repo root
sys.path.insert(0, '.')
import polars as pl
from _fixture import SCAN_DEF, bars_eager, bars_lazy

from scanlang import apply, score_bars, validate


## 1. The fixture

`bars_eager()` and `bars_lazy()` give the same 120-row frame
(60 days x 2 symbols). The shape you pick decides where your
`.collect()` lives.

In [2]:
df_eager = bars_eager()
df_lazy  = bars_lazy()
print(type(df_eager).__name__, df_eager.shape)
print(type(df_lazy).__name__,  df_lazy.collect().shape)

DataFrame (120, 7)
LazyFrame (120, 7)


## 2. Score every symbol's latest bar

`score_bars` is lazy in / lazy out. We collect once at the edge
so the rest of this notebook stays in eager land.

In [3]:
scored = score_bars(df_eager).collect()
scored.select("symbol", "session", "close", "score", "phase")

symbol,session,close,score,phase
str,date,f64,i16,str
"""AAA""",2026-03-01,69.0,60,"""BASE"""
"""BBB""",2026-03-01,1.0,20,"""NONE"""


## 3. Define and validate a scan

`validate` returns `[]` when the dict is structurally sound; a
non-empty list means the dict has an unknown property, a wrong
operator, or a mismatched dtype. No exception is raised here.

In [4]:
errors = validate(SCAN_DEF)
print("errors:", errors)  # []

errors: []


## 4. Apply the scan

`apply` is shape-preserving: eager in -> eager out. With a
`LazyFrame` it stays lazy until you say so.

In [5]:
picks = apply(scored, SCAN_DEF)
picks.select("symbol", "score", "phase")

symbol,score,phase
str,i16,str
"""AAA""",60,"""BASE"""


## 5. Same scan, lazy end-to-end

No intermediate `.collect()`. One `.collect()` at the end --
the lazy contract.

In [6]:
lazy_picks = apply(score_bars(df_lazy), SCAN_DEF)
assert isinstance(lazy_picks, pl.LazyFrame)
lazy_picks.select("symbol", "score", "phase").collect()

symbol,score,phase
str,i16,str
"""AAA""",60,"""BASE"""


## 6. Lock the behaviour

These assertions are what makes the notebook CI-friendly:
`nbconvert --execute` will fail loudly if a future change
breaks the happy path.

In [7]:
assert errors == []
assert picks.height == 1
assert picks["symbol"][0] == "AAA"
assert picks["score"][0] == scored["score"].max()
print("01_first_scan OK")

01_first_scan OK


## Where to next

- [`02_first_scan_marimo.py`](./02_first_scan_marimo.py) -- same
  scan in a marimo notebook (reactive cells, headless `marimo
  export html`).
- Tutorial: [First scan in 5 minutes](../tutorials/first-scan.md)
- How-to: [Eager vs lazy frames](../how-to/eager-frames.md)
- Examples: [docs/examples/](../examples/)
